# Downloading necessary packages and connecting to MongoDB

In [29]:
# Downloading all necessary packages
import pandas as pd
from pymongo import MongoClient

In [30]:
# Connecting to MongoDB
client = MongoClient("mongodb://localhost:27017/")
db = client["literary_trends"]

# Critic's Pick Cleaning

In [31]:
# Loading critic's picks into a dataframe
critics_data = list(db["critics_picks"].find())
critics_df = pd.DataFrame(critics_data)

print(critics_df.shape)
print(critics_df.columns.tolist())
critics_df.head()

(113549, 20)
['_id', 'abstract', 'web_url', 'snippet', 'lead_paragraph', 'print_section', 'print_page', 'source', 'multimedia', 'headline', 'keywords', 'pub_date', 'document_type', 'news_desk', 'section_name', 'byline', 'type_of_material', 'word_count', 'uri', 'subsection_name']


,_id,abstract,web_url,snippet,lead_paragraph,print_section,print_page,source,multimedia,headline,keywords,pub_date,document_type,news_desk,section_name,byline,type_of_material,word_count,uri,subsection_name
0,nyt://article/0f501db5-3d93-5ea9-afbb-b043b1e5...,Ann Powers reviews performance by rap group Ar...,https://www.nytimes.com/2000/01/01/arts/pop-re...,Ann Powers reviews performance by rap group Ar...,"Q-Unique, one of the rappers in the Bushwick-b...",F,22,The New York Times,[],{'main': 'Rappers Who Won't Let The Audience B...,"[{'name': 'organizations', 'value': 'ARSONISTS...",2000-01-01T05:00:00+0000,article,The Arts/Cultural Desk,Arts,"{'original': 'By Ann Powers', 'person': [{'fir...",Review,393,nyt://article/0f501db5-3d93-5ea9-afbb-b043b1e5...,NaN
1,nyt://article/78d0f9c0-3052-59fa-b086-fbd8a3ed...,Jennifer Dunning reviews Eliot Feld Ballet's p...,https://www.nytimes.com/2000/01/01/arts/dance-...,Jennifer Dunning reviews Eliot Feld Ballet's p...,The glorious athletes of Eliot Feld's ''Aurora...,F,10,The New York Times,[],{'main': 'Two Idealized Worlds With Contrastin...,"[{'name': 'organizations', 'value': 'FELD, ELI...",2000-01-01T05:00:00+0000,article,The Arts/Cultural Desk,Arts,"{'original': 'By Jennifer Dunning', 'person': ...",Review,330,nyt://article/78d0f9c0-3052-59fa-b086-fbd8a3ed...,NaN
2,nyt://article/af41fef5-ae8c-5120-9711-b1ca69cb...,Jennifer Dunning reviews performance by Alvin ...,https://www.nytimes.com/2000/01/01/arts/dance-...,Jennifer Dunning reviews performance by Alvin ...,One of the happiest aspects of the Alvin Ailey...,F,8,The New York Times,[],{'main': 'Lovers Who Burn the Air Around Them'...,"[{'name': 'organizations', 'value': 'AILEY, AL...",2000-01-01T05:00:00+0000,article,The Arts/Cultural Desk,Arts,"{'original': 'By Jennifer Dunning', 'person': ...",Review,350,nyt://article/af41fef5-ae8c-5120-9711-b1ca69cb...,NaN
3,nyt://article/c1e8332c-4e14-5292-86cc-cb1770fb...,Anita Gates reviews play Inappropriate by A Mi...,https://www.nytimes.com/2000/01/01/theater/the...,Anita Gates reviews play Inappropriate by A Mi...,''Inappropriate'' is not ''Rent.'' A theatergo...,F,28,The New York Times,[],"{'main': 'Smells Like Teen Spirit, or Whatever...","[{'name': 'persons', 'value': 'McNeil, Lonnie'...",2000-01-01T05:00:00+0000,article,The Arts/Cultural Desk,Theater,"{'original': 'By Anita Gates', 'person': [{'fi...",Review,790,nyt://article/c1e8332c-4e14-5292-86cc-cb1770fb...,NaN
4,nyt://article/c5a5176b-b65d-5a3c-a20e-8391a3f8...,Jack Anderson reviews performance by Forces of...,https://www.nytimes.com/2000/01/01/arts/dance-...,Jack Anderson reviews performance by Forces of...,"Forces of Nature, a company trained in various...",F,18,The New York Times,[],"{'main': 'The Traditions of Africa, Mixed With...","[{'name': 'organizations', 'value': 'Forces of...",2000-01-01T05:00:00+0000,article,The Arts/Cultural Desk,Arts,"{'original': 'By Jack Anderson', 'person': [{'...",Review,275,nyt://article/c5a5176b-b65d-5a3c-a20e-8391a3f8...,NaN


In [32]:
# Keeping only relevant columns
critics_clean = critics_df[[
    'headline',
    'byline',
    'section_name',
    'pub_date',
    'abstract',
    'web_url',
    'keywords'
]].copy()

# Extracting headline text
critics_clean['headline'] = critics_clean['headline'].apply(
    lambda x: x.get('main') if isinstance(x, dict) else x
)

# Extracting byline text
critics_clean['byline'] = critics_clean['byline'].apply(
    lambda x: x.get('original') if isinstance(x, dict) else x
)

# Removing "By " from byline
critics_clean['byline'] = critics_clean['byline'].str.replace('By ', '', regex=False)

# Cleaning up pub_date
critics_clean['pub_date'] = pd.to_datetime(critics_clean['pub_date'], format='mixed').dt.date

# Extracting keywords as comma-separated string
critics_clean['keywords'] = critics_clean['keywords'].apply(
    lambda x: ', '.join([kw.get('value', '') for kw in x]) if isinstance(x, list) else x
)

critics_clean = critics_clean.dropna()
print(f"Critics rows after dropping nulls: {len(critics_clean)}")
critics_clean.head()

Critics rows after dropping nulls: 113548


,headline,byline,section_name,pub_date,abstract,web_url,keywords
0,Rappers Who Won't Let The Audience Be Lazy,Ann Powers,Arts,2000-01-01,Ann Powers reviews performance by rap group Ar...,https://www.nytimes.com/2000/01/01/arts/pop-re...,"ARSONISTS, Music, Reviews"
1,Two Idealized Worlds With Contrasting Moods,Jennifer Dunning,Arts,2000-01-01,Jennifer Dunning reviews Eliot Feld Ballet's p...,https://www.nytimes.com/2000/01/01/arts/dance-...,"FELD, ELIOT, BALLET, Reviews, Dancing"
2,Lovers Who Burn the Air Around Them,Jennifer Dunning,Arts,2000-01-01,Jennifer Dunning reviews performance by Alvin ...,https://www.nytimes.com/2000/01/01/arts/dance-...,"AILEY, ALVIN, AMERICAN DANCE THEATER, Reviews,..."
3,"Smells Like Teen Spirit, or Whatever",Anita Gates,Theater,2000-01-01,Anita Gates reviews play Inappropriate by A Mi...,https://www.nytimes.com/2000/01/01/theater/the...,"McNeil, Lonnie, DeSisto, A Michael, Reviews, T..."
4,"The Traditions of Africa, Mixed With the Modern",Jack Anderson,Arts,2000-01-01,Jack Anderson reviews performance by Forces of...,https://www.nytimes.com/2000/01/01/arts/dance-...,"Forces of Nature, Reviews, Dancing"


# Bestsellers Cleaning

In [33]:
# Loading critic's picks into a dataframe
bestsellers_data = list(db["bestsellers"].find())
bestsellers_df = pd.DataFrame(bestsellers_data)

print(bestsellers_df.shape)
print(bestsellers_df.columns.tolist())
bestsellers_df.head()

(13815, 31)
['_id', 'age_group', 'amazon_product_url', 'article_chapter_link', 'asterisk', 'author', 'book_image', 'book_image_height', 'book_image_width', 'book_review_link', 'book_uri', 'contributor', 'contributor_note', 'created_date', 'dagger', 'description', 'first_chapter_link', 'price', 'primary_isbn10', 'primary_isbn13', 'publisher', 'rank', 'rank_last_week', 'sunday_review_link', 'title', 'updated_date', 'weeks_on_list', 'isbns', 'buy_links', 'list_name', 'list_date']


,_id,age_group,amazon_product_url,article_chapter_link,asterisk,author,book_image,book_image_height,book_image_width,book_review_link,...,rank,rank_last_week,sunday_review_link,title,updated_date,weeks_on_list,isbns,buy_links,list_name,list_date
0,69df8660e0828eb3aa0969c5,,http://www.amazon.com/Rogue-Lawyer-John-Grisha...,,0,John Grisham,https://static01.nyt.com/bestsellers/images/97...,495,326,,...,1,1,,ROGUE LAWYER,2026-03-18T00:05:12.62Z,8,"[{'isbn10': '', 'isbn13': '9780385539432'}]","[{'name': 'Amazon', 'url': 'http://www.amazon....",hardcover-fiction,2016-01-01
1,69df8660e0828eb3aa0969c6,,http://www.amazon.com/Cross-Justice-Alex-James...,,0,James Patterson,https://static01.nyt.com/bestsellers/images/97...,495,318,,...,2,2,,CROSS JUSTICE,2026-03-07T04:25:51.865Z,3,"[{'isbn10': '', 'isbn13': '9780316407045'}]","[{'name': 'Amazon', 'url': 'http://www.amazon....",hardcover-fiction,2016-01-01
2,69df8660e0828eb3aa0969c7,,http://www.amazon.com/See-Me-Nicholas-Sparks/d...,,0,Nicholas Sparks,https://static01.nyt.com/bestsellers/images/97...,495,328,,...,3,6,,SEE ME,2026-03-18T00:05:12.347Z,9,"[{'isbn10': '', 'isbn13': '9781455520619'}]","[{'name': 'Amazon', 'url': 'http://www.amazon....",hardcover-fiction,2016-01-01
3,69df8660e0828eb3aa0969c8,,http://www.amazon.com/The-Bazaar-Bad-Dreams-St...,,0,Stephen King,https://static01.nyt.com/bestsellers/images/97...,495,326,,...,4,4,,THE BAZAAR OF BAD DREAMS,2026-03-18T00:05:10.991Z,6,"[{'isbn10': '', 'isbn13': '9781501111679'}]","[{'name': 'Amazon', 'url': 'http://www.amazon....",hardcover-fiction,2016-01-01
4,69df8660e0828eb3aa0969c9,,http://www.amazon.com/All-Light-We-Cannot-See/...,,0,Anthony Doerr,https://static01.nyt.com/bestsellers/images/97...,192,128,,...,5,8,https://www.nytimes.com/2014/05/11/books/revie...,ALL THE LIGHT WE CANNOT SEE,2026-03-18T00:25:04.931Z,84,"[{'isbn10': '', 'isbn13': '9781476746586'}]","[{'name': 'Amazon', 'url': 'http://www.amazon....",hardcover-fiction,2016-01-01


In [34]:
# Keeping only relevant columns
bestsellers_clean = bestsellers_df[[
    'title',
    'author',
    'publisher',
    'description',
    'rank',
    'weeks_on_list',
    'primary_isbn13',
    'list_name',
    'list_date'
]].copy()

# Clean up list_date to just the date
bestsellers_clean['list_date'] = pd.to_datetime(bestsellers_clean['list_date'], format='mixed').dt.date

bestsellers_clean = bestsellers_clean.dropna()
print(f"Bestsellers rows after dropping nulls: {len(bestsellers_clean)}")
bestsellers_clean.head()

Bestsellers rows after dropping nulls: 13815


,title,author,publisher,description,rank,weeks_on_list,primary_isbn13,list_name,list_date
0,ROGUE LAWYER,John Grisham,Doubleday,The attorney Sebastian Rudd is a “lone gunman”...,1,8,9780385539432,hardcover-fiction,2016-01-01
1,CROSS JUSTICE,James Patterson,"Little, Brown","Detective Alex Cross returns to his hometown, ...",2,3,9780316407045,hardcover-fiction,2016-01-01
2,SEE ME,Nicholas Sparks,Grand Central,A couple in love are threatened by secrets fro...,3,9,9781455520619,hardcover-fiction,2016-01-01
3,THE BAZAAR OF BAD DREAMS,Stephen King,Scribner,"Twenty short stories, some never before publis...",4,6,9781501111679,hardcover-fiction,2016-01-01
4,ALL THE LIGHT WE CANNOT SEE,Anthony Doerr,Scribner,The lives of a blind French girl and a gadget-...,5,84,9781476746586,hardcover-fiction,2016-01-01


# Exporting both df as .csv files

In [35]:
critics_clean.to_csv('critics_clean.csv', index=False)
bestsellers_clean.to_csv('bestsellers_clean.csv', index=False)